# 🥇 Gold — Payment breakdown

**Purpose.** Aggregate trips by payment type. The TLC encodes payment as a small integer (1=Credit Card, 2=Cash, 3=No charge, 4=Dispute, 5=Unknown, 6=Voided). This Gold table joins in human-readable labels and computes both trip counts and revenue share.

**Source:** `workspace.silver.yellow_taxi`
**Output:** `workspace.gold.payment_breakdown` (~5 rows)

In [0]:
from pyspark.sql.functions import col, count, sum as spark_sum, avg, round as spark_round, when, lit

SOURCE_TABLE = "workspace.silver.yellow_taxi"
FULL_TABLE_NAME = "workspace.gold.payment_breakdown"

silver_df = spark.table(SOURCE_TABLE)

# Map the integer payment_type codes to human-readable labels.
labelled_df = silver_df.withColumn(
    "payment_method",
    when(col("payment_type") == 1, lit("Credit card"))
    .when(col("payment_type") == 2, lit("Cash"))
    .when(col("payment_type") == 3, lit("No charge"))
    .when(col("payment_type") == 4, lit("Dispute"))
    .when(col("payment_type") == 5, lit("Unknown"))
    .when(col("payment_type") == 6, lit("Voided"))
    .otherwise(lit("Other")),
)

payment_df = (
    labelled_df
    .groupBy("payment_type", "payment_method")
    .agg(
        count("*").alias("trip_count"),
        spark_round(spark_sum("total_amount"), 2).alias("total_revenue"),
        spark_round(avg("fare_amount"), 2).alias("avg_fare"),
        spark_round(avg("tip_amount"), 2).alias("avg_tip"),
    )
    .orderBy(col("trip_count").desc())
)

row_count = payment_df.count()
print(f"Payment breakdown: {row_count} rows")

(
    payment_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(FULL_TABLE_NAME)
)
print(f"✓ Wrote {row_count} rows to {FULL_TABLE_NAME}")

In [0]:
result = spark.table(FULL_TABLE_NAME)

print("Payment breakdown:")
display(result.orderBy(col("trip_count").desc()))

# Compute share percentages.
total = result.agg({"trip_count": "sum"}).first()[0]
print(f"\nTotal trips across all payment types: {total:,}")

print("\nShare by payment method:")
from pyspark.sql.functions import expr
display(
    result
    .withColumn("trip_share_pct", spark_round(col("trip_count") / total * 100, 1))
    .select("payment_method", "trip_count", "trip_share_pct", "avg_fare", "avg_tip")
    .orderBy(col("trip_count").desc())
)